In [1]:
import os, random, argparse, itertools, numpy as np, scipy.io as sio
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score, silhouette_score
from sklearn.cluster import KMeans
from sklearn.preprocessing import scale
from _utils import MMDataset, clustering_acc, overall_performance_report

class DUCMME(nn.Module):
    def __init__(self, embed_dim=200, num_samples=10000, num_views=3, feature_dims=[1000, 1000, 500], hidden_dims=[512, 512, 512], n_clusters=10, alpha=1.0):
        super(DUCMME, self).__init__()
        self.embed_dim = embed_dim; self.num_samples = num_samples; self.num_views = num_views; self.feature_dims = feature_dims; self.hidden_dims = hidden_dims; self.n_clusters = n_clusters; self.alpha = alpha
        # 1. Multi-view Feature Extraction by Fusion-Net
        self.fusion_net_encoder = nn.ModuleList([nn.Sequential(nn.Linear(feature_dims[i], hidden_dims[i]), nn.BatchNorm1d(hidden_dims[i]), nn.ReLU(),
                                                               nn.Linear(hidden_dims[i], embed_dim)) for i in range(num_views)]) # encode each view
        self.fusion_net_mha = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=10, batch_first=True) # batch_first=True: (batch_size, seq_len, hidden_dim)
        self.fusion_net_linear = nn.Linear(3*embed_dim, embed_dim) # linear projection of the fused encoded features
        # 2. Uncertainty-Aware Reconstruction by Reconstruction-Net and Uncertainty-Net
        self.reconstruct_net_list = nn.ModuleList([nn.Sequential(nn.Linear(self.embed_dim, hidden_dims[i]), nn.BatchNorm1d(hidden_dims[i]), nn.ReLU(), 
                                                                 nn.Linear(hidden_dims[i], feature_dims[i])) for i in range(num_views)]) # reconstruct each view
        self.uncertainty_net_list = nn.ModuleList([nn.Sequential(nn.Linear(self.embed_dim, hidden_dims[i]), nn.BatchNorm1d(hidden_dims[i]), nn.ReLU(), 
                                                                 nn.Linear(hidden_dims[i], feature_dims[i])) for i in range(num_views)]) # predict uncertainty for each view
        # 3. Deep Embedding Clustering by DEC
        self._cluster_centers = nn.Parameter(torch.Tensor(self.n_clusters, self.embed_dim))
        nn.init.xavier_uniform_(self._cluster_centers.data)
        
    def forward_embedding(self, x):
        encoded_output_list = [self.fusion_net_encoder[i](x[i]) for i in range(self.num_views)] # encode each view
        encoded_output_list = torch.stack(encoded_output_list, dim=1) # stack the encoded features from all views, (batch_size, num_views, embed_dim)
        encoded_output_list, _ = self.fusion_net_mha(encoded_output_list, encoded_output_list, encoded_output_list) # fuse the encoded features from all views by a multihead attention, (batch_size, num_views, embed_dim)
        encoded_output_list = encoded_output_list.contiguous().view(encoded_output_list.shape[0], -1) # flatten the encoded features, (batch_size, num_views*embed_dim)
        embedding = self.fusion_net_linear(encoded_output_list) # linear projection of the fused encoded features
        return embedding # get the embedding of the latent space H, (batch_size, embed_dim)

    def forward_uncertainty_aware_reconstruction(self, x):
        embedding = self.forward_embedding(x) # shape: [batch_size, embed_dim]
        reconstructions = [self.reconstruct_net_list[i](embedding) for i in range(self.num_views)] # reconstruct each view
        uncertainties = [self.uncertainty_net_list[i](embedding) for i in range(self.num_views)] # predict uncertainty for each view
        return reconstructions, uncertainties
        
    def forward_similarity_matrix_q(self, x): # calculate the similarity matrix q using t-distribution
        embedding = self.forward_embedding(x) # shape: [batch_size, embed_dim]
        q = 1.0 / (1.0 + torch.sum((embedding.unsqueeze(1) - self._cluster_centers) ** 2, dim=2) / self.alpha) # shape: [batch_size, n_clusters]
        q = q ** ((self.alpha + 1.0) / 2.0) # , shape: [batch_size, n_clusters]
        q = q / torch.sum(q, dim=1, keepdim=True) # Normalize q to sum to 1 across clusters, shape: [batch_size, n_clusters]
        return q, embedding # q can be regarded as the probability of the sample belonging to each cluster
    
    @property
    def cluster_centers(self):
        return self._cluster_centers.data.detach().cpu().numpy() # shape: (n_clusters, embed_dim)
    
    @cluster_centers.setter
    def cluster_centers(self, centers): # shape: (n_clusters, embed_dim)
        centers = torch.tensor(centers, dtype=torch.float32, device=self._cluster_centers.device)
        self._cluster_centers.data.copy_(centers) # copy the cluster centers to the model, set the cluster centers to the new cluster centers
        
    @staticmethod
    def target_distribution(q):
        weight = q ** 2 / torch.sum(q, dim=0) # shape: [batch_size, n_clusters]
        p = weight / torch.sum(weight, dim=1, keepdim=True) # Normalize p to sum to 1 across clusters, shape: [batch_size, n_clusters]
        return p.clone().detach()
    
    def reconstruction_loss(self, x):
        x_rec, _ = self.forward_uncertainty_aware_reconstruction(x) # reconstruct each view and predict uncertainty
        return sum([F.mse_loss(x_rec[v], x[v], reduction='mean') for v in range(self.num_views)]) # sum the losses from all views
    
    def uncertainty_aware_reconstruction_loss(self, x):
        x_rec, log_sigma_2 = self.forward_uncertainty_aware_reconstruction(x) # reconstruct each view and predict uncertainty
        # Clip log_sigma_2 to prevent numerical instability (exp(-log_sigma_2) overflow/underflow)
        # Clamp to reasonable range: -10 to 10, which corresponds to sigma^2 from exp(-10) to exp(10)
        # log_sigma_2 = [torch.clamp(log_sigma_2[v], min=-10.0, max=10.0) for v in range(self.num_views)] # shape: [num_views, batch_size, feature_dim] for numerically stable computation
        return sum([0.5 * torch.mean((x_rec[v] - x[v])**2 * torch.exp(-log_sigma_2[v]) + log_sigma_2[v]) for v in range(self.num_views)]) # uncertainty is equal to log_sigma_2
    
    def clustering_loss(self, x, p):
        q, _ = self.forward_similarity_matrix_q(x) # shape: [batch_size, n_clusters]
        return F.kl_div(q.log(), p, reduction='batchmean') # shape: ()

In [2]:
random.seed(0); np.random.seed(0); torch.manual_seed(0); torch.cuda.manual_seed_all(0) # Set random seed for reproducibility
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dataset = MMDataset('./data/data_bulk_multiomics/BRCA/', concat_data=False)
data = [x.clone().to(device) for x in dataset.X]; label = dataset.Y.clone().numpy()
data_views = dataset.data_views; data_samples = dataset.data_samples; data_features = dataset.data_features; data_categories = dataset.categories
label_dict = dataset.get_label_to_name()

dataloader = torch.utils.data.DataLoader(dataset, batch_size=data_samples, shuffle=True)
model = DUCMME(embed_dim=20, feature_dims=data_features, num_views=data_views, hidden_dims=[512, 512, 512], num_samples=data_samples, n_clusters=data_categories, alpha=1.0).to(device)
## === Stage 1: Uncertainty-Aware Reconstruction Pretraining ===
print("\n=== Stage 1: Uncertainty-Aware Reconstruction Pretraining ===")
print("Basic reconstruction training...")
optimizer = torch.optim.Adam(model.parameters(), lr=5e-3)
losses_convergence_plot_reconstruction = []
for epoch in range(200):
    model.train()
    losses = []
    for batch_idx, (x, y, idx) in enumerate(dataloader):
        x = [x.to(device) for x in x]
        optimizer.zero_grad()
        loss = model.reconstruction_loss(x)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    losses_convergence_plot_reconstruction.append(np.mean(losses))
    print(f'Epoch {epoch} completed. Average Loss: {np.mean(losses):.4f}')
embedding = model.forward_embedding(data).detach().cpu().numpy() # shape: [num_samples, embed_dim]
kmeans = KMeans(n_clusters=data_categories, n_init=20, random_state=0)
preds = kmeans.fit_predict(embedding)
acc_val = clustering_acc(label, preds)
nmi_val = normalized_mutual_info_score(label, preds)
asw_val = silhouette_score(embedding, preds)
ari_val = adjusted_rand_score(label, preds)
print(f"Pretraining completed. ACC: {acc_val:.4f}, NMI: {nmi_val:.4f}, ASW: {asw_val:.4f}, ARI: {ari_val:.4f}")
print("Uncertainty-aware reconstruction finetuning...")
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
losses_convergence_plot_reconstruction_finetuning = []
for epoch in range(100):
    model.train()
    losses = []
    for batch_idx, (x, y, idx) in enumerate(dataloader):
        x = [x.to(device) for x in x]
        optimizer.zero_grad()
        loss = model.uncertainty_aware_reconstruction_loss(x)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    losses_convergence_plot_reconstruction_finetuning.append(np.mean(losses))
    print(f'Epoch {epoch} completed. Average Loss: {np.mean(losses):.4f}')
embedding = model.forward_embedding(data).detach().cpu().numpy() # shape: [num_samples, embed_dim]
kmeans = KMeans(n_clusters=data_categories, n_init=20, random_state=0)
preds = kmeans.fit_predict(embedding)
acc_val = clustering_acc(label, preds)
nmi_val = normalized_mutual_info_score(label, preds)
asw_val = silhouette_score(embedding, preds)
ari_val = adjusted_rand_score(label, preds)
print(f"Finetuning completed. ACC: {acc_val:.4f}, NMI: {nmi_val:.4f}, ASW: {asw_val:.4f}, ARI: {ari_val:.4f}")

## === Stage 2: Deep Embedding Clustering by DEC ===
print("\n=== Stage 2: Deep Embedding Clustering ===")
print("Cluster center initialization...")
model.eval()
embedding = model.forward_embedding(data).detach().cpu().numpy() # shape: [num_samples, embed_dim]
kmeans = KMeans(n_clusters=data_categories, n_init=20, random_state=0)
preds = kmeans.fit_predict(embedding)
acc_val = clustering_acc(label, preds)
nmi_val = normalized_mutual_info_score(label, preds)
asw_val = silhouette_score(embedding, preds)
ari_val = adjusted_rand_score(label, preds)
print(f"Cluster center initialization completed. ACC: {acc_val:.4f}, NMI: {nmi_val:.4f}, ASW: {asw_val:.4f}, ARI: {ari_val:.4f}")
model.cluster_centers = kmeans.cluster_centers_ # shape: (n_clusters, embed_dim)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.9)
losses = []
acc_convergence_plot = []
nmi_convergence_plot = []
asw_convergence_plot = []
ari_convergence_plot = []
embeddings_dict = {}
preds_dict = {}
for epoch in range(150):
    # Update target distribution periodically
    if epoch % 1 == 0:
        model.eval()
        with torch.no_grad():
            q, embedding = model.forward_similarity_matrix_q(data)
            p = model.target_distribution(q)
        y_pred = torch.argmax(q, dim=1).cpu().numpy()
        acc_val = clustering_acc(label, y_pred)
        nmi_val = normalized_mutual_info_score(label, y_pred)
        asw_val = silhouette_score(embedding.cpu().numpy(), y_pred)
        ari_val = adjusted_rand_score(label, y_pred)
        acc_convergence_plot.append(acc_val)
        nmi_convergence_plot.append(nmi_val)
        asw_convergence_plot.append(asw_val)
        ari_convergence_plot.append(ari_val)
        embeddings_dict[epoch] = embedding
        preds_dict[epoch] = y_pred
        if epoch == 0:
            delta_label = 1.0
            y_pred_last = y_pred.copy()
            print(f'[Epoch {epoch}] ACC: {acc_val:.4f}, NMI: {nmi_val:.4f}, ASW: {asw_val:.4f}, ARI: {ari_val:.4f}, Delta: {delta_label:.4f}')
        else:
            delta_label = np.sum(y_pred != y_pred_last).astype(np.float32) / y_pred.shape[0]
            y_pred_last = y_pred.copy()
            print(f'[Epoch {epoch}] ACC: {acc_val:.4f}, NMI: {nmi_val:.4f}, ASW: {asw_val:.4f}, ARI: {ari_val:.4f}, Delta: {delta_label:.4f}')
            if delta_label < 0.001:
                convergence_epoch = epoch
                print('Converged, stopping training...')
                # break
    # Training based on the target distribution that is updated periodically
    model.train()
    losses = []
    for batch_idx, (x, y, idx) in enumerate(dataloader):
        x = [x.to(device) for x in x]
        optimizer.zero_grad()
        loss = model.clustering_loss(x, p[idx])
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    scheduler.step()
    print(f'Epoch {epoch} completed. Average Loss: {np.mean(losses):.4f}')
print(f'Final ACC: {clustering_acc(label, y_pred):.4f}')

modality_mrna shape: (875, 1000)
modality_meth shape: (875, 1000)
modality_mirna shape: (875, 503)

=== Stage 1: Uncertainty-Aware Reconstruction Pretraining ===
Basic reconstruction training...
Epoch 0 completed. Average Loss: 1.2580
Epoch 1 completed. Average Loss: 1.5399
Epoch 2 completed. Average Loss: 0.8419
Epoch 3 completed. Average Loss: 0.4985
Epoch 4 completed. Average Loss: 0.5469
Epoch 5 completed. Average Loss: 0.5280
Epoch 6 completed. Average Loss: 0.3862
Epoch 7 completed. Average Loss: 0.2977
Epoch 8 completed. Average Loss: 0.2993
Epoch 9 completed. Average Loss: 0.2999
Epoch 10 completed. Average Loss: 0.2584
Epoch 11 completed. Average Loss: 0.2003
Epoch 12 completed. Average Loss: 0.1771
Epoch 13 completed. Average Loss: 0.1800
Epoch 14 completed. Average Loss: 0.1798
Epoch 15 completed. Average Loss: 0.1592
Epoch 16 completed. Average Loss: 0.1298
Epoch 17 completed. Average Loss: 0.1149
Epoch 18 completed. Average Loss: 0.1149
Epoch 19 completed. Average Loss: 0.

In [ ]:
# modality_mrna shape: (875, 1000)
# modality_meth shape: (875, 1000)
# modality_mirna shape: (875, 503)
# === Stage 1: Uncertainty-Aware Reconstruction Pretraining ===
# Basic reconstruction training...
# Pretraining completed. ACC: 0.5566, NMI: 0.3751, ASW: 0.4607, ARI: 0.2674
# Uncertainty-aware reconstruction finetuning...
# Finetuning completed. ACC: 0.5657, NMI: 0.3996, ASW: 0.4749, ARI: 0.2621
# === Stage 2: Deep Embedding Clustering ===
# Cluster center initialization...
# Cluster center initialization completed. ACC: 0.5726, NMI: 0.4015, ASW: 0.4758, ARI: 0.2641
# [Epoch 0] ACC: 0.5726, NMI: 0.4015, ASW: 0.4758, ARI: 0.2641, Delta: 1.0000
# Epoch 0 completed. Average Loss: 0.1136
# [Epoch 1] ACC: 0.6091, NMI: 0.3963, ASW: 0.4558, ARI: 0.2695, Delta: 0.1406
# Epoch 1 completed. Average Loss: 0.1281
# [Epoch 2] ACC: 0.6160, NMI: 0.3867, ASW: 0.4217, ARI: 0.2875, Delta: 0.1646 # select this result
# Epoch 2 completed. Average Loss: 0.1551
# [Epoch 3] ACC: 0.5840, NMI: 0.4006, ASW: 0.3901, ARI: 0.2914, Delta: 0.1509
# Epoch 3 completed. Average Loss: 0.1704
# [Epoch 4] ACC: 0.5291, NMI: 0.3957, ASW: 0.5253, ARI: 0.2567, Delta: 0.1109
# Epoch 4 completed. Average Loss: 0.1877
# [Epoch 5] ACC: 0.5726, NMI: 0.3853, ASW: 0.4450, ARI: 0.2415, Delta: 0.1189
# Epoch 5 completed. Average Loss: 0.2161
# [Epoch 6] ACC: 0.6103, NMI: 0.3862, ASW: 0.3047, ARI: 0.2782, Delta: 0.1931
# Epoch 6 completed. Average Loss: 0.2686
# [Epoch 7] ACC: 0.5931, NMI: 0.4095, ASW: 0.2974, ARI: 0.2941, Delta: 0.1383
# Epoch 7 completed. Average Loss: 0.3219
# [Epoch 8] ACC: 0.5657, NMI: 0.4025, ASW: 0.5906, ARI: 0.2752, Delta: 0.0343
# Epoch 8 completed. Average Loss: 0.3323
# [Epoch 9] ACC: 0.5269, NMI: 0.3918, ASW: 0.5324, ARI: 0.2564, Delta: 0.0469
# Epoch 9 completed. Average Loss: 0.3344
# [Epoch 10] ACC: 0.5314, NMI: 0.3830, ASW: 0.4581, ARI: 0.2508, Delta: 0.0606
# Epoch 10 completed. Average Loss: 0.3541
# [Epoch 11] ACC: 0.5886, NMI: 0.3796, ASW: 0.3636, ARI: 0.2672, Delta: 0.0709
# Epoch 11 completed. Average Loss: 0.3849
# [Epoch 12] ACC: 0.6400, NMI: 0.3796, ASW: 0.2421, ARI: 0.3042, Delta: 0.0926 # select this result
# Epoch 12 completed. Average Loss: 0.4105
# [Epoch 13] ACC: 0.6549, NMI: 0.3545, ASW: 0.1276, ARI: 0.3084, Delta: 0.0994
# Epoch 13 completed. Average Loss: 0.4140
# [Epoch 14] ACC: 0.6606, NMI: 0.3656, ASW: 0.0554, ARI: 0.3017, Delta: 0.0800
# Epoch 14 completed. Average Loss: 0.3737
# [Epoch 15] ACC: 0.6309, NMI: 0.3195, ASW: 0.0328, ARI: 0.2467, Delta: 0.0526
# Epoch 15 completed. Average Loss: 0.2928
# [Epoch 16] ACC: 0.6183, NMI: 0.2974, ASW: 0.0746, ARI: 0.2168, Delta: 0.0206
# Epoch 16 completed. Average Loss: 0.2011
# [Epoch 17] ACC: 0.6126, NMI: 0.2917, ASW: 0.2713, ARI: 0.2024, Delta: 0.0103
# Epoch 17 completed. Average Loss: 0.1303
# [Epoch 18] ACC: 0.5943, NMI: 0.2390, ASW: 0.2486, ARI: 0.1566, Delta: 0.0229
# Epoch 18 completed. Average Loss: 0.0902
# [Epoch 19] ACC: 0.5646, NMI: 0.1626, ASW: 0.2162, ARI: 0.0978, Delta: 0.0320
# Epoch 19 completed. Average Loss: 0.0704
# [Epoch 20] ACC: 0.5440, NMI: 0.1130, ASW: 0.1926, ARI: 0.0624, Delta: 0.0229
# Epoch 20 completed. Average Loss: 0.0589
# [Epoch 21] ACC: 0.5383, NMI: 0.0994, ASW: 0.1869, ARI: 0.0506, Delta: 0.0080
# Epoch 21 completed. Average Loss: 0.0497
# [Epoch 22] ACC: 0.5371, NMI: 0.1009, ASW: 0.1816, ARI: 0.0481, Delta: 0.0023
# Epoch 22 completed. Average Loss: 0.0414
# [Epoch 23] ACC: 0.5360, NMI: 0.0979, ASW: 0.1787, ARI: 0.0465, Delta: 0.0011
# Epoch 23 completed. Average Loss: 0.0340
# [Epoch 24] ACC: 0.5349, NMI: 0.0950, ASW: 0.1756, ARI: 0.0449, Delta: 0.0011
# Epoch 24 completed. Average Loss: 0.0278
# [Epoch 25] ACC: 0.5349, NMI: 0.0950, ASW: 0.1736, ARI: 0.0449, Delta: 0.0000
# Converged, stopping training...
# Epoch 25 completed. Average Loss: 0.0227
# [Epoch 26] ACC: 0.5349, NMI: 0.0950, ASW: 0.1723, ARI: 0.0449, Delta: 0.0000
# Converged, stopping training...
# Epoch 26 completed. Average Loss: 0.0187
# [Epoch 27] ACC: 0.5326, NMI: 0.0891, ASW: 0.1700, ARI: 0.0418, Delta: 0.0046
# Epoch 27 completed. Average Loss: 0.0154
# [Epoch 28] ACC: 0.5314, NMI: 0.0861, ASW: 0.1677, ARI: 0.0403, Delta: 0.0011
# Epoch 28 completed. Average Loss: 0.0129
# [Epoch 29] ACC: 0.5303, NMI: 0.0832, ASW: 0.1632, ARI: 0.0387, Delta: 0.0011
# Epoch 29 completed. Average Loss: 0.0108
# [Epoch 30] ACC: 0.5291, NMI: 0.0802, ASW: 0.1575, ARI: 0.0372, Delta: 0.0011
# Epoch 30 completed. Average Loss: 0.0091
# [Epoch 31] ACC: 0.5291, NMI: 0.0802, ASW: 0.1520, ARI: 0.0372, Delta: 0.0000
# Converged, stopping training...
# Epoch 31 completed. Average Loss: 0.0078
# [Epoch 32] ACC: 0.5257, NMI: 0.0714, ASW: 0.1441, ARI: 0.0328, Delta: 0.0057
# Epoch 32 completed. Average Loss: 0.0066
# [Epoch 33] ACC: 0.5200, NMI: 0.0567, ASW: 0.1337, ARI: 0.0257, Delta: 0.0057
# Epoch 33 completed. Average Loss: 0.0057
# [Epoch 34] ACC: 0.5200, NMI: 0.0567, ASW: 0.1289, ARI: 0.0257, Delta: 0.0000
# Converged, stopping training...
# Epoch 34 completed. Average Loss: 0.0050
# [Epoch 35] ACC: 0.5154, NMI: 0.0450, ASW: 0.1161, ARI: 0.0202, Delta: 0.0046
# Epoch 35 completed. Average Loss: 0.0043
# [Epoch 36] ACC: 0.5109, NMI: 0.0334, ASW: 0.1038, ARI: 0.0150, Delta: 0.0046
# Epoch 36 completed. Average Loss: 0.0038
# [Epoch 37] ACC: 0.5040, NMI: 0.0161, ASW: 0.0971, ARI: 0.0076, Delta: 0.0069
# Epoch 37 completed. Average Loss: 0.0033
# [Epoch 38] ACC: 0.5040, NMI: 0.0161, ASW: 0.1099, ARI: 0.0076, Delta: 0.0023
# Epoch 38 completed. Average Loss: 0.0030
# [Epoch 39] ACC: 0.5040, NMI: 0.0161, ASW: 0.1111, ARI: 0.0076, Delta: 0.0000
# Converged, stopping training...
# Epoch 39 completed. Average Loss: 0.0026
# [Epoch 40] ACC: 0.5017, NMI: 0.0106, ASW: 0.1048, ARI: 0.0052, Delta: 0.0023
# Epoch 40 completed. Average Loss: 0.0024
# [Epoch 41] ACC: 0.5017, NMI: 0.0106, ASW: 0.1039, ARI: 0.0052, Delta: 0.0000
# Converged, stopping training...
# Epoch 41 completed. Average Loss: 0.0022
# [Epoch 42] ACC: 0.5006, NMI: 0.0081, ASW: 0.1085, ARI: 0.0041, Delta: 0.0011
# Epoch 42 completed. Average Loss: 0.0020
# [Epoch 43] ACC: 0.5006, NMI: 0.0081, ASW: 0.1068, ARI: 0.0041, Delta: 0.0000
# Converged, stopping training...
# Epoch 43 completed. Average Loss: 0.0018
# [Epoch 44] ACC: 0.5006, NMI: 0.0081, ASW: 0.1050, ARI: 0.0041, Delta: 0.0000
# Converged, stopping training...
# Epoch 44 completed. Average Loss: 0.0017
# [Epoch 45] ACC: 0.5006, NMI: 0.0081, ASW: 0.1033, ARI: 0.0041, Delta: 0.0000
# Converged, stopping training...
# Epoch 45 completed. Average Loss: 0.0016
# [Epoch 46] ACC: 0.5006, NMI: 0.0081, ASW: 0.1016, ARI: 0.0041, Delta: 0.0000
# Converged, stopping training...
# Epoch 46 completed. Average Loss: 0.0016
# [Epoch 47] ACC: 0.5017, NMI: 0.0106, ASW: 0.0878, ARI: 0.0052, Delta: 0.0011
# Epoch 47 completed. Average Loss: 0.0015
# [Epoch 48] ACC: 0.5029, NMI: 0.0133, ASW: 0.0854, ARI: 0.0064, Delta: 0.0011
# Epoch 48 completed. Average Loss: 0.0015
# [Epoch 49] ACC: 0.5029, NMI: 0.0133, ASW: 0.0839, ARI: 0.0064, Delta: 0.0000
# Converged, stopping training...
# Epoch 49 completed. Average Loss: 0.0014
# [Epoch 50] ACC: 0.5040, NMI: 0.0161, ASW: 0.0852, ARI: 0.0076, Delta: 0.0011
# Epoch 50 completed. Average Loss: 0.0014
# [Epoch 51] ACC: 0.5040, NMI: 0.0161, ASW: 0.0845, ARI: 0.0076, Delta: 0.0000
# Converged, stopping training...
# Epoch 51 completed. Average Loss: 0.0014
# [Epoch 52] ACC: 0.5051, NMI: 0.0189, ASW: 0.0840, ARI: 0.0088, Delta: 0.0011
# Epoch 52 completed. Average Loss: 0.0014
# [Epoch 53] ACC: 0.5063, NMI: 0.0218, ASW: 0.0804, ARI: 0.0100, Delta: 0.0011
# Epoch 53 completed. Average Loss: 0.0014
# [Epoch 54] ACC: 0.5097, NMI: 0.0304, ASW: 0.0749, ARI: 0.0138, Delta: 0.0034
# Epoch 54 completed. Average Loss: 0.0015
# [Epoch 55] ACC: 0.5097, NMI: 0.0304, ASW: 0.0737, ARI: 0.0138, Delta: 0.0000
# Converged, stopping training...
# Epoch 55 completed. Average Loss: 0.0015
# [Epoch 56] ACC: 0.5097, NMI: 0.0304, ASW: 0.0726, ARI: 0.0138, Delta: 0.0000
# Converged, stopping training...
# Epoch 56 completed. Average Loss: 0.0015
# [Epoch 57] ACC: 0.5097, NMI: 0.0304, ASW: 0.0716, ARI: 0.0138, Delta: 0.0000
# Converged, stopping training...
# Epoch 57 completed. Average Loss: 0.0015
# [Epoch 58] ACC: 0.5120, NMI: 0.0363, ASW: 0.0704, ARI: 0.0163, Delta: 0.0023
# Epoch 58 completed. Average Loss: 0.0016
# [Epoch 59] ACC: 0.5143, NMI: 0.0421, ASW: 0.0798, ARI: 0.0189, Delta: 0.0023
# Epoch 59 completed. Average Loss: 0.0016
# [Epoch 60] ACC: 0.5154, NMI: 0.0450, ASW: 0.0887, ARI: 0.0202, Delta: 0.0011
# Epoch 60 completed. Average Loss: 0.0017
# [Epoch 61] ACC: 0.5189, NMI: 0.0538, ASW: 0.0918, ARI: 0.0243, Delta: 0.0034
# Epoch 61 completed. Average Loss: 0.0017
# [Epoch 62] ACC: 0.5211, NMI: 0.0597, ASW: 0.0907, ARI: 0.0271, Delta: 0.0023
# Epoch 62 completed. Average Loss: 0.0017
# [Epoch 63] ACC: 0.5211, NMI: 0.0597, ASW: 0.0901, ARI: 0.0271, Delta: 0.0000
# Converged, stopping training...
# Epoch 63 completed. Average Loss: 0.0018
# [Epoch 64] ACC: 0.5223, NMI: 0.0626, ASW: 0.0890, ARI: 0.0285, Delta: 0.0011
# Epoch 64 completed. Average Loss: 0.0018
# [Epoch 65] ACC: 0.5246, NMI: 0.0685, ASW: 0.0912, ARI: 0.0313, Delta: 0.0023
# Epoch 65 completed. Average Loss: 0.0019
# [Epoch 66] ACC: 0.5291, NMI: 0.0800, ASW: 0.0978, ARI: 0.0390, Delta: 0.0057
# Epoch 66 completed. Average Loss: 0.0019
# [Epoch 67] ACC: 0.5303, NMI: 0.0829, ASW: 0.1012, ARI: 0.0405, Delta: 0.0011
# Epoch 67 completed. Average Loss: 0.0020
# [Epoch 68] ACC: 0.5326, NMI: 0.0887, ASW: 0.1010, ARI: 0.0436, Delta: 0.0023
# Epoch 68 completed. Average Loss: 0.0020
# [Epoch 69] ACC: 0.5337, NMI: 0.0916, ASW: 0.1031, ARI: 0.0451, Delta: 0.0011
# Epoch 69 completed. Average Loss: 0.0020
# [Epoch 70] ACC: 0.5371, NMI: 0.1003, ASW: 0.1060, ARI: 0.0499, Delta: 0.0034
# Epoch 70 completed. Average Loss: 0.0021
# [Epoch 71] ACC: 0.5394, NMI: 0.1061, ASW: 0.1064, ARI: 0.0531, Delta: 0.0023
# Epoch 71 completed. Average Loss: 0.0021
# [Epoch 72] ACC: 0.5440, NMI: 0.1179, ASW: 0.1110, ARI: 0.0597, Delta: 0.0046
# Epoch 72 completed. Average Loss: 0.0022
# [Epoch 73] ACC: 0.5520, NMI: 0.1382, ASW: 0.1228, ARI: 0.0736, Delta: 0.0091
# Epoch 73 completed. Average Loss: 0.0022
# [Epoch 74] ACC: 0.5554, NMI: 0.1471, ASW: 0.1269, ARI: 0.0791, Delta: 0.0034
# Epoch 74 completed. Average Loss: 0.0023
# [Epoch 75] ACC: 0.5554, NMI: 0.1471, ASW: 0.1265, ARI: 0.0791, Delta: 0.0000
# Converged, stopping training...
# Epoch 75 completed. Average Loss: 0.0023
# [Epoch 76] ACC: 0.5566, NMI: 0.1501, ASW: 0.1283, ARI: 0.0809, Delta: 0.0011
# Epoch 76 completed. Average Loss: 0.0023
# [Epoch 77] ACC: 0.5611, NMI: 0.1621, ASW: 0.1329, ARI: 0.0885, Delta: 0.0046
# Epoch 77 completed. Average Loss: 0.0024
# [Epoch 78] ACC: 0.5646, NMI: 0.1712, ASW: 0.1353, ARI: 0.0943, Delta: 0.0034
# Epoch 78 completed. Average Loss: 0.0024
# [Epoch 79] ACC: 0.5680, NMI: 0.1804, ASW: 0.1386, ARI: 0.1002, Delta: 0.0034
# Epoch 79 completed. Average Loss: 0.0025
# [Epoch 80] ACC: 0.5680, NMI: 0.1804, ASW: 0.1383, ARI: 0.1002, Delta: 0.0000
# Converged, stopping training...
# Epoch 80 completed. Average Loss: 0.0025
# [Epoch 81] ACC: 0.5703, NMI: 0.1866, ASW: 0.1417, ARI: 0.1042, Delta: 0.0023
# Epoch 81 completed. Average Loss: 0.0026
# [Epoch 82] ACC: 0.5703, NMI: 0.1866, ASW: 0.1415, ARI: 0.1042, Delta: 0.0000
# Converged, stopping training...
# Epoch 82 completed. Average Loss: 0.0026
# [Epoch 83] ACC: 0.5726, NMI: 0.1928, ASW: 0.1436, ARI: 0.1083, Delta: 0.0023
# Epoch 83 completed. Average Loss: 0.0026
# [Epoch 84] ACC: 0.5737, NMI: 0.1959, ASW: 0.1445, ARI: 0.1104, Delta: 0.0011
# Epoch 84 completed. Average Loss: 0.0027
# [Epoch 85] ACC: 0.5749, NMI: 0.1990, ASW: 0.1455, ARI: 0.1125, Delta: 0.0011
# Epoch 85 completed. Average Loss: 0.0027
# [Epoch 86] ACC: 0.5771, NMI: 0.2049, ASW: 0.1486, ARI: 0.1185, Delta: 0.0034
# Epoch 86 completed. Average Loss: 0.0028
# [Epoch 87] ACC: 0.5771, NMI: 0.2049, ASW: 0.1485, ARI: 0.1185, Delta: 0.0000
# Converged, stopping training...
# Epoch 87 completed. Average Loss: 0.0028
# [Epoch 88] ACC: 0.5771, NMI: 0.2049, ASW: 0.1484, ARI: 0.1185, Delta: 0.0000
# Converged, stopping training...
# Epoch 88 completed. Average Loss: 0.0029
# [Epoch 89] ACC: 0.5771, NMI: 0.2048, ASW: 0.1493, ARI: 0.1202, Delta: 0.0011
# Epoch 89 completed. Average Loss: 0.0029
# [Epoch 90] ACC: 0.5806, NMI: 0.2142, ASW: 0.1527, ARI: 0.1267, Delta: 0.0034
# Epoch 90 completed. Average Loss: 0.0030
# [Epoch 91] ACC: 0.5806, NMI: 0.2142, ASW: 0.1527, ARI: 0.1267, Delta: 0.0000
# Converged, stopping training...
# Epoch 91 completed. Average Loss: 0.0030
# [Epoch 92] ACC: 0.5829, NMI: 0.2206, ASW: 0.1554, ARI: 0.1311, Delta: 0.0023
# Epoch 92 completed. Average Loss: 0.0030
# [Epoch 93] ACC: 0.5829, NMI: 0.2206, ASW: 0.1554, ARI: 0.1311, Delta: 0.0000
# Converged, stopping training...
# Epoch 93 completed. Average Loss: 0.0031
# [Epoch 94] ACC: 0.5829, NMI: 0.2206, ASW: 0.1554, ARI: 0.1311, Delta: 0.0000
# Converged, stopping training...
# Epoch 94 completed. Average Loss: 0.0031
# [Epoch 95] ACC: 0.5840, NMI: 0.2238, ASW: 0.1567, ARI: 0.1333, Delta: 0.0011
# Epoch 95 completed. Average Loss: 0.0032
# [Epoch 96] ACC: 0.5886, NMI: 0.2368, ASW: 0.1616, ARI: 0.1423, Delta: 0.0046
# Epoch 96 completed. Average Loss: 0.0032
# [Epoch 97] ACC: 0.5897, NMI: 0.2401, ASW: 0.1628, ARI: 0.1446, Delta: 0.0011
# Epoch 97 completed. Average Loss: 0.0033
# [Epoch 98] ACC: 0.5897, NMI: 0.2401, ASW: 0.1629, ARI: 0.1446, Delta: 0.0000
# Converged, stopping training...
# Epoch 98 completed. Average Loss: 0.0033
# [Epoch 99] ACC: 0.5897, NMI: 0.2401, ASW: 0.1630, ARI: 0.1446, Delta: 0.0000
# Converged, stopping training...
# Epoch 99 completed. Average Loss: 0.0034
# [Epoch 100] ACC: 0.5909, NMI: 0.2434, ASW: 0.1645, ARI: 0.1469, Delta: 0.0011
# Epoch 100 completed. Average Loss: 0.0035
# [Epoch 101] ACC: 0.5909, NMI: 0.2433, ASW: 0.1658, ARI: 0.1486, Delta: 0.0011
# Epoch 101 completed. Average Loss: 0.0035
# [Epoch 102] ACC: 0.5920, NMI: 0.2466, ASW: 0.1672, ARI: 0.1510, Delta: 0.0011
# Epoch 102 completed. Average Loss: 0.0036
# [Epoch 103] ACC: 0.5920, NMI: 0.2466, ASW: 0.1674, ARI: 0.1510, Delta: 0.0000
# Converged, stopping training...
# Epoch 103 completed. Average Loss: 0.0036
# [Epoch 104] ACC: 0.5920, NMI: 0.2466, ASW: 0.1677, ARI: 0.1510, Delta: 0.0000
# Converged, stopping training...
# Epoch 104 completed. Average Loss: 0.0037
# [Epoch 105] ACC: 0.5920, NMI: 0.2466, ASW: 0.1680, ARI: 0.1510, Delta: 0.0000
# Converged, stopping training...
# Epoch 105 completed. Average Loss: 0.0037
# [Epoch 106] ACC: 0.5920, NMI: 0.2466, ASW: 0.1683, ARI: 0.1510, Delta: 0.0000
# Converged, stopping training...
# Epoch 106 completed. Average Loss: 0.0038
# [Epoch 107] ACC: 0.5920, NMI: 0.2466, ASW: 0.1686, ARI: 0.1510, Delta: 0.0000
# Converged, stopping training...
# Epoch 107 completed. Average Loss: 0.0038
# [Epoch 108] ACC: 0.5931, NMI: 0.2499, ASW: 0.1698, ARI: 0.1533, Delta: 0.0011
# Epoch 108 completed. Average Loss: 0.0039
# [Epoch 109] ACC: 0.5931, NMI: 0.2499, ASW: 0.1702, ARI: 0.1533, Delta: 0.0000
# Converged, stopping training...
# Epoch 109 completed. Average Loss: 0.0040
# [Epoch 110] ACC: 0.5931, NMI: 0.2499, ASW: 0.1705, ARI: 0.1533, Delta: 0.0000
# Converged, stopping training...
# Epoch 110 completed. Average Loss: 0.0040
# [Epoch 111] ACC: 0.5943, NMI: 0.2533, ASW: 0.1718, ARI: 0.1557, Delta: 0.0011
# Epoch 111 completed. Average Loss: 0.0041
# [Epoch 112] ACC: 0.5954, NMI: 0.2566, ASW: 0.1732, ARI: 0.1580, Delta: 0.0011
# Epoch 112 completed. Average Loss: 0.0042
# [Epoch 113] ACC: 0.5954, NMI: 0.2566, ASW: 0.1736, ARI: 0.1580, Delta: 0.0000
# Converged, stopping training...
# Epoch 113 completed. Average Loss: 0.0043
# [Epoch 114] ACC: 0.5954, NMI: 0.2566, ASW: 0.1741, ARI: 0.1580, Delta: 0.0000
# Converged, stopping training...
# Epoch 114 completed. Average Loss: 0.0043
# [Epoch 115] ACC: 0.5966, NMI: 0.2601, ASW: 0.1769, ARI: 0.1622, Delta: 0.0023
# Epoch 115 completed. Average Loss: 0.0044
# [Epoch 116] ACC: 0.5966, NMI: 0.2601, ASW: 0.1774, ARI: 0.1622, Delta: 0.0000
# Converged, stopping training...
# Epoch 116 completed. Average Loss: 0.0045
# [Epoch 117] ACC: 0.5966, NMI: 0.2601, ASW: 0.1779, ARI: 0.1622, Delta: 0.0000
# Converged, stopping training...
# Epoch 117 completed. Average Loss: 0.0046
# [Epoch 118] ACC: 0.5966, NMI: 0.2604, ASW: 0.1793, ARI: 0.1640, Delta: 0.0011
# Epoch 118 completed. Average Loss: 0.0047
# [Epoch 119] ACC: 0.5966, NMI: 0.2604, ASW: 0.1800, ARI: 0.1640, Delta: 0.0000
# Converged, stopping training...
# Epoch 119 completed. Average Loss: 0.0048
# [Epoch 120] ACC: 0.5977, NMI: 0.2581, ASW: 0.1821, ARI: 0.1672, Delta: 0.0023
# Epoch 120 completed. Average Loss: 0.0049
# [Epoch 121] ACC: 0.5989, NMI: 0.2615, ASW: 0.1839, ARI: 0.1696, Delta: 0.0011
# Epoch 121 completed. Average Loss: 0.0050
# [Epoch 122] ACC: 0.5989, NMI: 0.2619, ASW: 0.1854, ARI: 0.1714, Delta: 0.0011
# Epoch 122 completed. Average Loss: 0.0051
# [Epoch 123] ACC: 0.5989, NMI: 0.2619, ASW: 0.1865, ARI: 0.1714, Delta: 0.0000
# Converged, stopping training...
# Epoch 123 completed. Average Loss: 0.0051
# [Epoch 124] ACC: 0.6011, NMI: 0.2687, ASW: 0.1890, ARI: 0.1762, Delta: 0.0023
# Epoch 124 completed. Average Loss: 0.0052
# [Epoch 125] ACC: 0.6023, NMI: 0.2669, ASW: 0.1922, ARI: 0.1798, Delta: 0.0023
# Epoch 125 completed. Average Loss: 0.0054
# [Epoch 126] ACC: 0.6023, NMI: 0.2669, ASW: 0.1934, ARI: 0.1798, Delta: 0.0000
# Converged, stopping training...
# Epoch 126 completed. Average Loss: 0.0055
# [Epoch 127] ACC: 0.6023, NMI: 0.2669, ASW: 0.1946, ARI: 0.1798, Delta: 0.0000
# Converged, stopping training...
# Epoch 127 completed. Average Loss: 0.0056
# [Epoch 128] ACC: 0.6034, NMI: 0.2703, ASW: 0.1965, ARI: 0.1822, Delta: 0.0011
# Epoch 128 completed. Average Loss: 0.0057
# [Epoch 129] ACC: 0.6057, NMI: 0.2773, ASW: 0.1990, ARI: 0.1872, Delta: 0.0023
# Epoch 129 completed. Average Loss: 0.0058
# [Epoch 130] ACC: 0.6057, NMI: 0.2773, ASW: 0.2004, ARI: 0.1872, Delta: 0.0000
# Converged, stopping training...
# Epoch 130 completed. Average Loss: 0.0060
# [Epoch 131] ACC: 0.6057, NMI: 0.2773, ASW: 0.2019, ARI: 0.1872, Delta: 0.0000
# Converged, stopping training...
# Epoch 131 completed. Average Loss: 0.0061
# [Epoch 132] ACC: 0.6057, NMI: 0.2773, ASW: 0.2035, ARI: 0.1872, Delta: 0.0000
# Converged, stopping training...
# Epoch 132 completed. Average Loss: 0.0063
# [Epoch 133] ACC: 0.6057, NMI: 0.2778, ASW: 0.2060, ARI: 0.1890, Delta: 0.0011
# Epoch 133 completed. Average Loss: 0.0065
# [Epoch 134] ACC: 0.6057, NMI: 0.2778, ASW: 0.2079, ARI: 0.1890, Delta: 0.0000
# Converged, stopping training...
# Epoch 134 completed. Average Loss: 0.0067
# [Epoch 135] ACC: 0.6057, NMI: 0.2778, ASW: 0.2100, ARI: 0.1890, Delta: 0.0000
# Converged, stopping training...
# Epoch 135 completed. Average Loss: 0.0069
# [Epoch 136] ACC: 0.6057, NMI: 0.2778, ASW: 0.2123, ARI: 0.1890, Delta: 0.0000
# Converged, stopping training...
# Epoch 136 completed. Average Loss: 0.0071
# [Epoch 137] ACC: 0.6080, NMI: 0.2855, ASW: 0.2159, ARI: 0.1959, Delta: 0.0034
# Epoch 137 completed. Average Loss: 0.0073
# [Epoch 138] ACC: 0.6091, NMI: 0.2851, ASW: 0.2196, ARI: 0.1992, Delta: 0.0023
# Epoch 138 completed. Average Loss: 0.0076
# [Epoch 139] ACC: 0.6103, NMI: 0.2887, ASW: 0.2224, ARI: 0.2018, Delta: 0.0011
# Epoch 139 completed. Average Loss: 0.0079
# [Epoch 140] ACC: 0.6103, NMI: 0.2855, ASW: 0.2259, ARI: 0.2026, Delta: 0.0011
# Epoch 140 completed. Average Loss: 0.0082
# [Epoch 141] ACC: 0.6114, NMI: 0.2898, ASW: 0.2300, ARI: 0.2069, Delta: 0.0023
# Epoch 141 completed. Average Loss: 0.0085
# [Epoch 142] ACC: 0.6114, NMI: 0.2898, ASW: 0.2339, ARI: 0.2069, Delta: 0.0000
# Converged, stopping training...
# Epoch 142 completed. Average Loss: 0.0088
# [Epoch 143] ACC: 0.6114, NMI: 0.2870, ASW: 0.2388, ARI: 0.2077, Delta: 0.0011
# Epoch 143 completed. Average Loss: 0.0092
# [Epoch 144] ACC: 0.6137, NMI: 0.2926, ASW: 0.2445, ARI: 0.2155, Delta: 0.0046
# Epoch 144 completed. Average Loss: 0.0097
# [Epoch 145] ACC: 0.6149, NMI: 0.2963, ASW: 0.2498, ARI: 0.2182, Delta: 0.0011
# Epoch 145 completed. Average Loss: 0.0102
# [Epoch 146] ACC: 0.6160, NMI: 0.3000, ASW: 0.2538, ARI: 0.2208, Delta: 0.0011
# Epoch 146 completed. Average Loss: 0.0107
# [Epoch 147] ACC: 0.6171, NMI: 0.3045, ASW: 0.2583, ARI: 0.2253, Delta: 0.0023
# Epoch 147 completed. Average Loss: 0.0112
# [Epoch 148] ACC: 0.6183, NMI: 0.3070, ASW: 0.2637, ARI: 0.2306, Delta: 0.0034
# Epoch 148 completed. Average Loss: 0.0118
# [Epoch 149] ACC: 0.6194, NMI: 0.3107, ASW: 0.2671, ARI: 0.2332, Delta: 0.0011
# Epoch 149 completed. Average Loss: 0.0124
# Final ACC: 0.6194

In [1]:
# 计算 Epoch 1, 2, 3, 16, 17 的四个指标的均值和方差
import numpy as np

# 提取这些 epoch 的数据
epoch_data = {
    1: {'ACC': 0.6091, 'NMI': 0.3963, 'ASW': 0.4558, 'ARI': 0.2695},
    2: {'ACC': 0.6160, 'NMI': 0.3867, 'ASW': 0.4217, 'ARI': 0.2875},
    3: {'ACC': 0.5840, 'NMI': 0.4006, 'ASW': 0.3901, 'ARI': 0.2914},
    16: {'ACC': 0.6183, 'NMI': 0.2974, 'ASW': 0.0746, 'ARI': 0.2168},
    17: {'ACC': 0.6126, 'NMI': 0.2917, 'ASW': 0.2713, 'ARI': 0.2024}
}

# 提取每个指标的值
metrics = ['ACC', 'NMI', 'ASW', 'ARI']
results = {}

for metric in metrics:
    values = [epoch_data[epoch][metric] for epoch in [1, 2, 3, 16, 17]]
    mean_val = np.mean(values)
    var_val = np.var(values, ddof=0)  # 总体方差
    std_val = np.std(values, ddof=0)   # 总体标准差
    results[metric] = {
        'values': values,
        'mean': mean_val,
        'variance': var_val,
        'std': std_val
    }

# 汇总表格
print("=" * 60)
print("汇总表格:")
print("=" * 60)
print(f"{'指标':<8} {'均值':<12} {'方差':<12} {'标准差':<12}")
print("-" * 60)
for metric in metrics:
    print(f"{metric:<8} {results[metric]['mean']:<12.6f} {results[metric]['variance']:<12.6f} {results[metric]['std']:<12.6f}")


汇总表格:
指标       均值           方差           标准差         
------------------------------------------------------------
ACC      0.608000     0.000154     0.012397    
NMI      0.354540     0.002423     0.049221    
ASW      0.322700     0.019251     0.138748    
ARI      0.253520     0.001361     0.036896    
